# Clean the CMS hospital files

Turn the raw CMS downloads into tables we can analyze. No charts. No UCI.

**Words used here**

- **Facility:** one hospital. The ID is `facility_id` (CMS also calls this a CCN).
- **Measure:** one published score, such as heart-failure readmission.
- **Missing:** CMS wrote `Not Available` instead of a number. We make that blank. It is not zero.
- **Footnote:** CMS's reason a number is blank or limited (for example, too few cases).
- **Left join:** keep every hospital. Attach a score only if CMS published one.

**Steps**

1. Clean the hospital list. This is the backbone.
2. Clean readmission and return-day scores.
3. Keep a few patient-survey items (HCAHPS).
4. Keep emergency-department wait times.
5. Attach selected scores to each hospital.
6. Check IDs and save.

Raw files in `data/raw/cms/` are not edited.


In [47]:
from pathlib import Path

import numpy as np
import pandas as pd

In [48]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "processed").is_dir():
            return path
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from the "
        "repository folder or from notebooks/."
    )

PROJECT_ROOT = find_project_root(Path.cwd())

RAW_CMS_DIR = PROJECT_ROOT / "data" / "raw" / "cms"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(RAW_CMS_DIR)
print(RAW_CMS_DIR.exists())

c:\Users\Micaela\Documents\CODING\hospital-operations-analytics
c:\Users\Micaela\Documents\CODING\hospital-operations-analytics\data\raw\cms
True


## Helpers

`Not Available` and `Not Applicable` become blank. Footnotes stay as text so codes like `16, 23` still match the crosswalk.


In [49]:
# Placeholder strings CMS uses instead of a blank cell.
MISSING_TOKENS = {"not available", "not applicable", ""}

# Census region for peer groups. Territories are not ranked with states.
CENSUS_REGION = {
    "CT": "Northeast", "ME": "Northeast", "MA": "Northeast", "NH": "Northeast",
    "RI": "Northeast", "VT": "Northeast", "NJ": "Northeast", "NY": "Northeast",
    "PA": "Northeast",
    "IL": "Midwest", "IN": "Midwest", "MI": "Midwest", "OH": "Midwest",
    "WI": "Midwest", "IA": "Midwest", "KS": "Midwest", "MN": "Midwest",
    "MO": "Midwest", "NE": "Midwest", "ND": "Midwest", "SD": "Midwest",
    "DE": "South", "DC": "South", "FL": "South", "GA": "South", "MD": "South",
    "NC": "South", "SC": "South", "VA": "South", "WV": "South", "AL": "South",
    "KY": "South", "MS": "South", "TN": "South", "AR": "South", "LA": "South",
    "OK": "South", "TX": "South",
    "AZ": "West", "CO": "West", "ID": "West", "MT": "West", "NV": "West",
    "NM": "West", "UT": "West", "WY": "West", "AK": "West", "CA": "West",
    "HI": "West", "OR": "West", "WA": "West",
    "AS": "Territory", "GU": "Territory", "MP": "Territory",
    "PR": "Territory", "VI": "Territory",
}

# Unplanned measures we attach to the facility mart (KPI dictionary).
UNPLANNED_KEEP = [
    "READM_30_HF",
    "READM_30_PN",
    "READM_30_COPD",
    "READM_30_AMI",
    "EDAC_30_HF",
    "EDAC_30_PN",
]

# HCAHPS items we keep. The raw file has 68 measure IDs; most are item percents.
HCAHPS_KEEP = [
    "H_STAR_RATING",
    "H_HSP_RATING_STAR_RATING",
    "H_HSP_RATING_LINEAR_SCORE",
    "H_COMP_1_STAR_RATING",
    "H_COMP_1_LINEAR_SCORE",
]

# ED throughput KPI plus volume as context. Other TE measures stay in raw.
TIMELY_KEEP = ["EDV", "OP_18a", "OP_18b", "OP_18c", "OP_18d", "OP_22", "OP_23"]


def snake_case_columns(df: pd.DataFrame) -> pd.DataFrame:
    """CMS headers have spaces and slashes. Make them usable in pandas/SQL."""
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.replace(r"[^\w]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
        .str.lower()
    )
    return out


def recode_missing(df: pd.DataFrame, columns) -> pd.DataFrame:
    """Turn CMS placeholder strings into true missing. Never fill with 0."""
    out = df.copy()
    for col in columns:
        text = out[col].astype("string").str.strip()
        out[col] = text.mask(text.str.lower().isin(MISSING_TOKENS))
    return out


def to_numeric(df: pd.DataFrame, columns) -> pd.DataFrame:
    """Parse published numbers after placeholders are gone."""
    out = df.copy()
    for col in columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


# CMS used two spellings for the same idea ("Too Small" vs "too small").
COMPARE_DISPLAY = {
    "no different than the national rate": "No different than the national rate",
    "no different than expected": "No different than expected",
    "number of cases too small": "Number of cases too small",
    "worse than the national rate": "Worse than the national rate",
    "worse than expected": "Worse than expected",
    "better than the national rate": "Better than the national rate",
    "better than expected": "Better than expected",
    "average days per 100 discharges": "Average days per 100 discharges",
    "fewer days than average per 100 discharges": "Fewer days than average per 100 discharges",
    "more days than average per 100 discharges": "More days than average per 100 discharges",
}

EDV_DISPLAY = {
    "low": "Low",
    "medium": "Medium",
    "high": "High",
    "very high": "Very high",
}

## Hospital list

One row per hospital. Later files attach to this list by `facility_id`.


In [50]:
# Keep Facility ID as text so 010001 does not become 10001.
gi_raw = pd.read_csv(
    RAW_CMS_DIR / "Hospital_General_Information.csv",
    dtype=str,
)

print(gi_raw.shape)
gi_raw.head()

(5419, 38)


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,NaN,15,15,NaN,10,10,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,NaN,15,15,NaN,10,10,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,NaN,15,15,NaN,10,9,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,3,2,NaN,15,5,NaN,10,7,NaN
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,1,5,2,29,15,10,29,10,7,29


In [51]:
gi_raw.columns.tolist()

['Facility ID',
 'Facility Name',
 'Address',
 'City/Town',
 'State',
 'ZIP Code',
 'County/Parish',
 'Telephone Number',
 'Hospital Type',
 'Hospital Ownership',
 'Emergency Services',
 'Meets criteria for birthing friendly designation',
 'Hospital overall rating',
 'Hospital overall rating footnote',
 'MORT Group Measure Count',
 'Count of Facility MORT Measures',
 'Count of MORT Measures Better',
 'Count of MORT Measures No Different',
 'Count of MORT Measures Worse',
 'MORT Group Footnote',
 'Safety Group Measure Count',
 'Count of Facility Safety Measures',
 'Count of Safety Measures Better',
 'Count of Safety Measures No Different',
 'Count of Safety Measures Worse',
 'Safety Group Footnote',
 'READM Group Measure Count',
 'Count of Facility READM Measures',
 'Count of READM Measures Better',
 'Count of READM Measures No Different',
 'Count of READM Measures Worse',
 'READM Group Footnote',
 'Pt Exp Group Measure Count',
 'Count of Facility Pt Exp Measures',
 'Pt Exp Group Footno

In [52]:
gi_raw["Hospital Type"].value_counts(dropna=False)

Hospital Type
Acute Care Hospitals                    3107
Critical Access Hospitals               1382
Psychiatric                              624
Acute Care - Veterans Administration     132
Childrens                                 94
Rural Emergency Hospital                  43
Acute Care - Department of Defense        32
Long-term                                  5
Name: count, dtype: int64

In [53]:
gi_raw["Hospital overall rating"].value_counts(dropna=False)

Hospital overall rating
Not Available    2245
3                 985
4                 946
2                 661
5                 384
1                 198
Name: count, dtype: int64

In [54]:
gi = snake_case_columns(gi_raw)

# Shorter names for the fields we actually use.
gi = gi.rename(
    columns={
        "city_town": "city",
        "zip_code": "zip_code",
        "county_parish": "county",
        "telephone_number": "phone",
        "meets_criteria_for_birthing_friendly_designation": "birthing_friendly",
        "hospital_overall_rating": "overall_rating",
        "hospital_overall_rating_footnote": "overall_rating_footnote",
        "count_of_facility_readm_measures": "readm_n_measures",
        "count_of_readm_measures_better": "readm_n_better",
        "count_of_readm_measures_no_different": "readm_n_no_different",
        "count_of_readm_measures_worse": "readm_n_worse",
        "readm_group_footnote": "readm_group_footnote",
        "pt_exp_group_measure_count": "pt_exp_group_measure_count",
        "count_of_facility_pt_exp_measures": "pt_exp_n_measures",
        "pt_exp_group_footnote": "pt_exp_group_footnote",
    }
)

id_cols = ["facility_id", "facility_name", "address", "city", "state", "zip_code", "county", "phone"]
profile_cols = [
    "hospital_type",
    "hospital_ownership",
    "emergency_services",
    "birthing_friendly",
]
rating_cols = ["overall_rating"]
readm_count_cols = [
    "readm_group_measure_count",
    "readm_n_measures",
    "readm_n_better",
    "readm_n_no_different",
    "readm_n_worse",
]
footnote_cols = ["overall_rating_footnote", "readm_group_footnote", "pt_exp_group_footnote"]

keep_gi = id_cols + profile_cols + rating_cols + readm_count_cols + footnote_cols + [
    "pt_exp_group_measure_count",
    "pt_exp_n_measures",
]
gi = gi[keep_gi]

# Placeholders -> missing, then numbers become numbers.
gi = recode_missing(gi, rating_cols + readm_count_cols + ["pt_exp_group_measure_count", "pt_exp_n_measures", "birthing_friendly"])
gi = to_numeric(gi, rating_cols + readm_count_cols + ["pt_exp_group_measure_count", "pt_exp_n_measures"])

gi["facility_id"] = gi["facility_id"].str.strip()
gi["census_region"] = gi["state"].map(CENSUS_REGION)
# Match emergency_services Yes/No. Blank still means not designated, not "No."
gi["birthing_friendly"] = gi["birthing_friendly"].replace({"Y": "Yes"})
gi["hospital_type"] = gi["hospital_type"].replace({"Childrens": "Children's"})

gi.head()

,facility_id,facility_name,address,city,state,zip_code,county,phone,hospital_type,hospital_ownership,...,readm_n_measures,readm_n_better,readm_n_no_different,readm_n_worse,overall_rating_footnote,readm_group_footnote,pt_exp_group_footnote,pt_exp_group_measure_count,pt_exp_n_measures,census_region
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,11,1,9,1,NaN,NaN,NaN,15,15,South
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,9,1,8,0,NaN,NaN,NaN,15,15,South
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,9,1,8,0,NaN,NaN,NaN,15,15,South
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,5,0,3,2,NaN,NaN,NaN,15,5,South
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,8,1,5,2,29,29,29,15,10,South


In [55]:
# Validate: one row per hospital, 5,419 IDs, stars only 1-5 when present.
assert gi["facility_id"].nunique() == len(gi)
assert gi["facility_id"].duplicated().sum() == 0
assert gi["facility_id"].str.len().eq(6).all()
assert gi["overall_rating"].dropna().between(1, 5).all()
assert gi["census_region"].notna().all()
assert not gi["overall_rating"].astype("string").str.lower().isin(MISSING_TOKENS).any()

print(len(gi), "facilities")
print(gi["overall_rating"].notna().sum(), "have a star rating")
print(gi["census_region"].value_counts().to_string())

5419 facilities
3174 have a star rating
census_region
South        2072
Midwest      1537
West         1076
Northeast     669
Territory      65


## Readmission and return days

One row per hospital per measure. We clean every row. The wide hospital table only keeps the scores named in the KPI dictionary. `Hybrid_HWR` stays in the long file only. It is often blank.


In [56]:
unplanned_raw = pd.read_csv(
    RAW_CMS_DIR / "Unplanned_Hospital_Visits-Hospital.csv",
    dtype=str,
)

print(unplanned_raw.shape)
print(unplanned_raw["Measure ID"].value_counts().to_string())

(67060, 20)
Measure ID
EDAC_30_AMI          4790
EDAC_30_HF           4790
EDAC_30_PN           4790
Hybrid_HWR           4790
OP_32                4790
OP_35_ADM            4790
OP_35_ED             4790
OP_36                4790
READM_30_AMI         4790
READM_30_CABG        4790
READM_30_COPD        4790
READM_30_HF          4790
READM_30_HIP_KNEE    4790
READM_30_PN          4790


In [57]:
unplanned_raw["Compared to National"].value_counts(dropna=False)

Compared to National
Not Available                                 20597
No Different Than the National Rate           20581
Number of Cases Too Small                     13767
Average Days per 100 Discharges                6524
No Different than expected                     2577
Fewer Days Than Average per 100 Discharges      896
More Days Than Average per 100 Discharges       872
Number of cases too small                       842
Worse Than the National Rate                    171
Better Than the National Rate                    98
Better than expected                             87
Worse than expected                              48
Name: count, dtype: int64

In [58]:
unplanned = snake_case_columns(unplanned_raw)

# Drop address fields. They already live on the facility profile.
unplanned = unplanned[
    [
        "facility_id",
        "measure_id",
        "measure_name",
        "compared_to_national",
        "denominator",
        "score",
        "lower_estimate",
        "higher_estimate",
        "number_of_patients",
        "number_of_patients_returned",
        "footnote",
        "start_date",
        "end_date",
    ]
]

score_cols = [
    "denominator",
    "score",
    "lower_estimate",
    "higher_estimate",
    "number_of_patients",
    "number_of_patients_returned",
]
unplanned = recode_missing(unplanned, score_cols + ["compared_to_national"])
unplanned = to_numeric(unplanned, score_cols)
# One spelling per comparison label.
unplanned["compared_to_national"] = (
    unplanned["compared_to_national"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(COMPARE_DISPLAY)
)

unplanned["facility_id"] = unplanned["facility_id"].str.strip()
unplanned["start_date"] = pd.to_datetime(unplanned["start_date"], errors="coerce")
unplanned["end_date"] = pd.to_datetime(unplanned["end_date"], errors="coerce")

unplanned.head()

,facility_id,measure_id,measure_name,compared_to_national,denominator,score,lower_estimate,higher_estimate,number_of_patients,number_of_patients_returned,footnote,start_date,end_date
0,010001,EDAC_30_AMI,Hospital return days for heart attack patients,Average days per 100 discharges,242,-2.6,-47.1,74.7,235,51,NaN,2022-07-01,2025-06-30
1,010001,EDAC_30_HF,Hospital return days for heart failure patients,Average days per 100 discharges,659,-14.4,-59.7,52.9,540,173,NaN,2022-07-01,2025-06-30
2,010001,EDAC_30_PN,Hospital return days for pneumonia patients,Average days per 100 discharges,586,0.7,-35.1,50.0,530,126,NaN,2022-07-01,2025-06-30
3,010001,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4,2024-07-01,2025-06-30
4,010001,OP_32,Rate of unplanned hospital visits after colono...,No different than the national rate,234,12.7,9.5,17.0,<NA>,<NA>,NaN,2022-01-01,2024-12-31


In [59]:
# Validate: unique hospital × measure, all IDs exist on the backbone.
assert unplanned.duplicated(subset=["facility_id", "measure_id"]).sum() == 0
assert unplanned["facility_id"].isin(gi["facility_id"]).all()
assert not unplanned["score"].astype("string").str.lower().isin(MISSING_TOKENS).fillna(False).any()

print(len(unplanned), "measure rows")
print(unplanned["facility_id"].nunique(), "hospitals")
print("score present", unplanned["score"].notna().sum())

67060 measure rows
4790 hospitals
score present 31854


## Patient survey (HCAHPS)

HCAHPS is a standard survey of how patients rated the stay. The raw file has 68 items. We keep five:

- `H_STAR_RATING`: overall survey star
- `H_HSP_RATING_STAR_RATING` / `H_HSP_RATING_LINEAR_SCORE`: hospital rating
- `H_COMP_1_STAR_RATING` / `H_COMP_1_LINEAR_SCORE`: nurse communication

`Not Applicable` on the wrong column for that row is just how the file is shaped. It becomes blank.


In [60]:
hcahps_raw = pd.read_csv(
    RAW_CMS_DIR / "HCAHPS-Hospital.csv",
    dtype=str,
)

print(hcahps_raw.shape)
print("measure IDs", hcahps_raw["HCAHPS Measure ID"].nunique())

(325720, 22)
measure IDs 68


In [61]:
hcahps = snake_case_columns(hcahps_raw)
hcahps = hcahps[hcahps["hcahps_measure_id"].isin(HCAHPS_KEEP)].copy()

hcahps = hcahps[
    [
        "facility_id",
        "hcahps_measure_id",
        "hcahps_question",
        "patient_survey_star_rating",
        "patient_survey_star_rating_footnote",
        "hcahps_linear_mean_value",
        "number_of_completed_surveys",
        "number_of_completed_surveys_footnote",
        "survey_response_rate_percent",
        "start_date",
        "end_date",
    ]
]

hcahps_num = [
    "patient_survey_star_rating",
    "hcahps_linear_mean_value",
    "number_of_completed_surveys",
    "survey_response_rate_percent",
]
hcahps = recode_missing(hcahps, hcahps_num)
hcahps = to_numeric(hcahps, hcahps_num)

hcahps["facility_id"] = hcahps["facility_id"].str.strip()
hcahps["start_date"] = pd.to_datetime(hcahps["start_date"], errors="coerce")
hcahps["end_date"] = pd.to_datetime(hcahps["end_date"], errors="coerce")

print(hcahps["hcahps_measure_id"].value_counts().to_string())
hcahps.head()

hcahps_measure_id
H_COMP_1_LINEAR_SCORE        4790
H_COMP_1_STAR_RATING         4790
H_HSP_RATING_LINEAR_SCORE    4790
H_HSP_RATING_STAR_RATING     4790
H_STAR_RATING                4790


,facility_id,hcahps_measure_id,hcahps_question,patient_survey_star_rating,patient_survey_star_rating_footnote,hcahps_linear_mean_value,number_of_completed_surveys,number_of_completed_surveys_footnote,survey_response_rate_percent,start_date,end_date
3,010001,H_COMP_1_LINEAR_SCORE,Nurse communication - linear mean score,<NA>,NaN,91,1392,NaN,17,2024-10-01,2025-09-30
4,010001,H_COMP_1_STAR_RATING,Nurse communication - star rating,3,NaN,<NA>,1392,NaN,17,2024-10-01,2025-09-30
60,010001,H_HSP_RATING_LINEAR_SCORE,Overall hospital rating - linear mean score,<NA>,NaN,90,1392,NaN,17,2024-10-01,2025-09-30
61,010001,H_HSP_RATING_STAR_RATING,Overall hospital rating - star rating,4,NaN,<NA>,1392,NaN,17,2024-10-01,2025-09-30
67,010001,H_STAR_RATING,Summary star rating,4,NaN,<NA>,1392,NaN,17,2024-10-01,2025-09-30


In [62]:
assert set(hcahps["hcahps_measure_id"]) == set(HCAHPS_KEEP)
assert hcahps.duplicated(subset=["facility_id", "hcahps_measure_id"]).sum() == 0
assert hcahps["facility_id"].isin(gi["facility_id"]).all()

print(len(hcahps), "selected HCAHPS rows")

23950 selected HCAHPS rows


## Emergency department times

We keep ED rows only.

- `OP_18b`: typical minutes in the ED before leaving (excludes transfers and psych visits). Lower is better. This is the wait-time KPI.
- `EDV`: how busy the ED is (very high / high / medium / low). A filter, not a KPI.


In [63]:
timely_raw = pd.read_csv(
    RAW_CMS_DIR / "Timely_and_Effective_Care-Hospital.csv",
    dtype=str,
)

print(timely_raw["Condition"].value_counts().to_string())

Condition
Electronic Clinical Quality Measure    68214
Emergency Department                   32606
Sepsis Care                            23290
Healthcare Personnel Vaccination        4658
Colonoscopy care                        4658
Cataract surgery outcome                4658


In [64]:
timely = snake_case_columns(timely_raw)
timely = timely[timely["measure_id"].isin(TIMELY_KEEP)].copy()

timely = timely[
    [
        "facility_id",
        "condition",
        "measure_id",
        "measure_name",
        "score",
        "sample",
        "footnote",
        "start_date",
        "end_date",
    ]
]

# EDV score is a volume label (very high, high, ...). OP_* scores are minutes.
timely = recode_missing(timely, ["score", "sample"])
timely["sample"] = pd.to_numeric(timely["sample"], errors="coerce")
ed_numeric = timely["measure_id"].ne("EDV")
timely["score"] = timely["score"].astype(object)
timely.loc[ed_numeric, "score"] = pd.to_numeric(
    timely.loc[ed_numeric, "score"], errors="coerce"
)

timely["facility_id"] = timely["facility_id"].str.strip()
timely["start_date"] = pd.to_datetime(timely["start_date"], errors="coerce")
timely["end_date"] = pd.to_datetime(timely["end_date"], errors="coerce")
# ED volume labels: Low / Medium / High / Very high.
is_edv = timely["measure_id"].eq("EDV")
timely.loc[is_edv, "score"] = (
    timely.loc[is_edv, "score"].astype("string").str.strip().str.lower().map(EDV_DISPLAY)
)

print(timely["measure_id"].value_counts().to_string())
timely.head()

measure_id
EDV       4658
OP_18a    4658
OP_18b    4658
OP_18c    4658
OP_18d    4658
OP_22     4658
OP_23     4658


,facility_id,condition,measure_id,measure_name,score,sample,footnote,start_date,end_date
0,010001,Emergency Department,EDV,Emergency department volume,Very high,<NA>,NaN,2024-01-01,2024-12-31
10,010001,Emergency Department,OP_18a,Average (median) time all patients spent in th...,214.0,405,NaN,2024-10-01,2025-09-30
11,010001,Emergency Department,OP_18b,Average (median) time patients spent in the em...,212.0,396,NaN,2024-10-01,2025-09-30
12,010001,Emergency Department,OP_18c,Average (median) time psychiatric/mental healt...,NaN,<NA>,1,2024-10-01,2025-09-30
13,010001,Emergency Department,OP_18d,Average (median) time patients spent in the em...,NaN,<NA>,1,2024-10-01,2025-09-30


In [65]:
assert timely.duplicated(subset=["facility_id", "measure_id"]).sum() == 0
assert timely["facility_id"].isin(gi["facility_id"]).all()

print(len(timely), "ED measure rows")

32606 ED measure rows


## Footnotes and measure dates

Lookups. `footnote` explains a blank. `measure_dates` is the date range CMS used for each score.


In [66]:
footnotes = snake_case_columns(
    pd.read_csv(RAW_CMS_DIR / "Footnote_Crosswalk.csv", dtype=str)
)
footnotes["footnote"] = footnotes["footnote"].str.strip()
footnotes["footnote_text"] = footnotes["footnote_text"].str.strip()

measure_dates = snake_case_columns(
    pd.read_csv(RAW_CMS_DIR / "Measure_Dates.csv", dtype=str)
)
measure_dates["start_date"] = pd.to_datetime(measure_dates["start_date"], errors="coerce")
measure_dates["end_date"] = pd.to_datetime(measure_dates["end_date"], errors="coerce")

assert footnotes["footnote"].nunique() == len(footnotes)
assert measure_dates["measure_id"].nunique() == len(measure_dates)

print(len(footnotes), "footnotes")
print(len(measure_dates), "measure date rows")
footnotes.head()

32 footnotes
171 measure date rows


,footnote,footnote_text
0,1,The number of cases/patients is too few to rep...
1,2,Data submitted were based on a sample of cases...
2,3,Results are based on a shorter time period tha...
3,4,Data suppressed by CMS for one or more quarters.
4,5,Results are not available for this reporting p...


## Hospital table (wide)

Start from the hospital list. Attach selected scores. If CMS did not publish a score, that cell stays blank. Do not treat blank as zero.


In [67]:
def pivot_unplanned(df: pd.DataFrame, measure_ids: list[str]) -> pd.DataFrame:
    """One row per hospital, score/comparison/interval/footnote per measure."""
    piece = df[df["measure_id"].isin(measure_ids)].copy()
    frames = []
    for mid, grp in piece.groupby("measure_id", sort=False):
        slug = mid.lower()
        wide = grp.set_index("facility_id")[
            ["score", "compared_to_national", "denominator", "lower_estimate", "higher_estimate", "footnote"]
        ].rename(
            columns={
                "score": f"{slug}_score",
                "compared_to_national": f"{slug}_compared",
                "denominator": f"{slug}_denominator",
                "lower_estimate": f"{slug}_lower",
                "higher_estimate": f"{slug}_higher",
                "footnote": f"{slug}_footnote",
            }
        )
        frames.append(wide)
    return pd.concat(frames, axis=1)


def pivot_hcahps(df: pd.DataFrame) -> pd.DataFrame:
    """Keep the field that exists for that item. Star rows have no linear score."""
    frames = []
    for mid, grp in df.groupby("hcahps_measure_id", sort=False):
        slug = mid.lower()
        if mid.endswith("_LINEAR_SCORE"):
            wide = grp.set_index("facility_id")[
                ["hcahps_linear_mean_value", "number_of_completed_surveys"]
            ].rename(
                columns={
                    "hcahps_linear_mean_value": slug,
                    "number_of_completed_surveys": f"{slug}_n_surveys",
                }
            )
        else:
            wide = grp.set_index("facility_id")[
                ["patient_survey_star_rating", "number_of_completed_surveys"]
            ].rename(
                columns={
                    "patient_survey_star_rating": slug,
                    "number_of_completed_surveys": f"{slug}_n_surveys",
                }
            )
        frames.append(wide)
    return pd.concat(frames, axis=1)


def pivot_timely(df: pd.DataFrame, measure_ids: list[str]) -> pd.DataFrame:
    piece = df[df["measure_id"].isin(measure_ids)].copy()
    frames = []
    for mid, grp in piece.groupby("measure_id", sort=False):
        slug = mid.lower()
        wide = grp.set_index("facility_id")[["score", "footnote"]].rename(
            columns={"score": f"{slug}_score", "footnote": f"{slug}_footnote"}
        )
        frames.append(wide)
    return pd.concat(frames, axis=1)


facility_mart = gi.copy()

# Left joins: missing KPI cells mean "not published," not zero.
facility_mart = facility_mart.merge(
    pivot_unplanned(unplanned, UNPLANNED_KEEP),
    how="left",
    left_on="facility_id",
    right_index=True,
    validate="one_to_one",
)
facility_mart = facility_mart.merge(
    pivot_hcahps(hcahps),
    how="left",
    left_on="facility_id",
    right_index=True,
    validate="one_to_one",
)
facility_mart = facility_mart.merge(
    pivot_timely(timely, ["EDV", "OP_18b"]),
    how="left",
    left_on="facility_id",
    right_index=True,
    validate="one_to_one",
)

print(facility_mart.shape)
facility_mart.head()

(5419, 74)


,facility_id,facility_name,address,city,state,zip_code,county,phone,hospital_type,hospital_ownership,...,h_hsp_rating_linear_score,h_hsp_rating_linear_score_n_surveys,h_hsp_rating_star_rating,h_hsp_rating_star_rating_n_surveys,h_star_rating,h_star_rating_n_surveys,edv_score,edv_footnote,op_18b_score,op_18b_footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,90,1392,4,1392,4,1392,Very high,NaN,212.0,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,87,745,3,745,3,745,Very high,NaN,143.0,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,84,1781,2,1781,2,1781,High,NaN,147.0,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,<NA>,84,<NA>,84,<NA>,84,Low,NaN,130.0,NaN
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,87,1203,3,1203,3,1203,NaN,5,158.0,NaN


In [68]:
# Mart must stay one row per backbone hospital.
assert len(facility_mart) == len(gi)
assert facility_mart["facility_id"].nunique() == len(facility_mart)
assert facility_mart["facility_id"].tolist() == gi["facility_id"].tolist()

print("HF readmission score present", facility_mart["readm_30_hf_score"].notna().sum())
print("OP_18b present", facility_mart["op_18b_score"].notna().sum())
print("HCAHPS overall star present", facility_mart["h_hsp_rating_star_rating"].notna().sum())

HF readmission score present 3253
OP_18b present 4081
HCAHPS overall star present 3183


## Save

Long tables keep one row per score. The wide table is for later charts. Rebuild from raw by re-running this notebook.


In [69]:
paths = {
    "cms_facility_profile.csv": gi,
    "cms_unplanned_mart.csv": unplanned,
    "cms_hcahps_mart.csv": hcahps,
    "cms_timely_ed_mart.csv": timely,
    "cms_footnote_crosswalk.csv": footnotes,
    "cms_measure_dates.csv": measure_dates,
    "cms_facility_mart.csv": facility_mart,
}

for name, df in paths.items():
    out = PROCESSED_DATA_DIR / name
    df.to_csv(out, index=False)
    print(out.name, df.shape)

cms_facility_profile.csv (5419, 24)
cms_unplanned_mart.csv (67060, 13)
cms_hcahps_mart.csv (23950, 11)
cms_timely_ed_mart.csv (32606, 9)
cms_footnote_crosswalk.csv (32, 2)
cms_measure_dates.csv (171, 6)
cms_facility_mart.csv (5419, 74)


## Outputs

| File | One row is | What it holds |
|---|---|---|
| `cms_facility_profile.csv` | A hospital | Type, ownership, region, stars |
| `cms_unplanned_mart.csv` | A hospital + one readmission/return score | All of those measures |
| `cms_hcahps_mart.csv` | A hospital + one survey item | The five HCAHPS items |
| `cms_timely_ed_mart.csv` | A hospital + one ED measure | ED times and volume |
| `cms_facility_mart.csv` | A hospital | Profile plus selected scores, side by side |
| `cms_footnote_crosswalk.csv` | A footnote code | Plain-language reason |
| `cms_measure_dates.csv` | A measure | Date range |

Next: notebook `03`, UCI stays.
